# Calculate actin metrics around compaction events

Per-image computation of mean LifeAct intensity in pixel populations defined by compaction-state transitions in time-lapse microscopy. Metrics are written back into the analysis dataframe (`df_path`) and a derived-mask OME-TIFF is saved per image for visual QC.

## Vocabulary

- *compact*: pixel inside the compaction segmentation (`SEG_CMP_CH`).
- *noncompact*: pixel CAAX-positive (`SEG_CAAX_CH`); equivalently, pixel not inside the compaction segmentation.
- *compact next frame* (0 → 1): pixel noncompact at *t* and compact at *t+1*.
- *noncompact next frame* (0 → 0): pixel noncompact at *t* and at *t+1*.
- *uncompact* (1 → 0): pixel compact at *t-1* and noncompact at *t*.

## Inputs

Each ROI image is an OME-TIFF stack with channels:
- `ACTIN_CH`: background-subtracted, ratio-corrected LifeAct.
- `SEG_CMP_CH`: binary segmentation of compaction zones.
- `SEG_CAAX_CH`: binary segmentation of CAAX-positive (noncompact) area.

The dataframe is expected to have one row per (image, frame), keyed by `ROI imgname` and `UID`, with a numeric `time interval` column (hours per frame).

## Per-frame columns written

- `mean actin int, compact next frame` — actin at *t* over 0 → 1 transition pixels.
- `mean actin int, noncompact next frame` — actin at *t* over pixels noncompact at both *t* and *t+1*.
- `mean actin int leading up to cmp` and `mean actin int leading up to cmp (never compact control)` — trajectory aligned to first-compaction events. Compactor cohort = pixels with at least `LOOKBACK_TIME` of pre-event history. Never-compact control = always-noncompact pixels within `NEIGHBOR_RADIUS_PX` of a valid compactor, paired in time with their nearest compactor (controls for photobleaching and local position effects).
- `time relative to cmp (hr)` — relative-time axis for the leading-up-to-cmp trajectories (negative hours before first compaction; 0 at the event).
- `mean actin int leading up to uncmp` — trajectory aligned to first-uncompaction events (1 → 0). With default lookback = 1 frame, this column holds two values per pixel: actin at *t-1* (still compact) and actin at *t* (just uncompacted).
- `time relative to uncmp (hr)` — relative-time axis for the leading-up-to-uncmp trajectory.

Diagnostic counts: `n pixels (compactor)`, `n pixels (never compact)`, `n pixels (uncompactor)`.

## Downstream steps in this notebook

1. Per-cell normalization: every actin column is divided by the cell’s mean LifeAct intensity. Normalized columns are added with the `norm ` prefix; original columns are preserved.
2. Trajectory plots of normalized leading-up-to-cmp (compactor vs control) and leading-up-to-uncmp.


## Imports

In [ ]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.ndimage import distance_transform_edt
from tqdm import tqdm

from bioio import BioImage
import bioio_ome_tiff
from bioio.writers import OmeTiffWriter

import microscopy_analysis.d00_utils.utilities as utils

## Plot style

In [ ]:
rc = {
    'svg.fonttype': 'none',
    'font.family': 'Arial',
    'figure.figsize': (4.5, 5),
    'figure.dpi': 150,
    'axes.linewidth': 0.5,
    'axes.labelweight': 'bold',
    'axes.labelsize': 10,
    'axes.labelpad': 7.5,
}
sns.set(rc)
sns.set_style('ticks')

ctrl_color = '#dd8452'
lat_color = '#4c72b0'
tx_palette = [ctrl_color, lat_color]

caax_color = '#A218A2'
cmp_color = '#116F11'
reg_palette = [caax_color, cmp_color]

## Configuration

Paths, channel indices, and the spatial/temporal windows used by the helpers.

In [ ]:
df_path = Path(
    '/Users/kwu2/Library/CloudStorage/Box-Box/Z-lab_Box/Z-lab shared folders/Kathryn + Eduardo/CE030_1st data set_JLY-LatA-Jasp-treatment/img_processing/tables/analysis.csv'
)

input_dirpath = Path(
    '/Users/kwu2/Library/CloudStorage/Box-Box/Z-lab_Box/Z-lab shared folders/Kathryn + Eduardo/CE030_1st data set_JLY-LatA-Jasp-treatment/img_processing/labkit_seg/caax_cell_bgsbratiocorractin_seg'
)
output_dirpath = input_dirpath.parent / 'corractin_seg_addlregions'
output_dirpath.mkdir(exist_ok=True)

# Channel indices in the source OME-TIFF
ACTIN_CH = 2
SEG_CMP_CH = 3
SEG_CAAX_CH = 4

# Lookback window for the leading-up-to-cmp metric, in TIME units (hours).
# Per-image lookback in frames is computed at runtime from each image's
# 'time interval' column (handles datasets with mixed acquisition rates).
# Pixels without this much pre-event history are excluded so the contributing
# cohort is identical at every relative-time slot.
LOOKBACK_TIME = 5  # hours

# Spatial radius (pixels) used to define the never compact control pool.
# Each always-CAAX+ pixel within this Euclidean distance of some compactor
# pixel becomes a control, paired in time with its single nearest compactor.
NEIGHBOR_RADIUS_PX = 50

# RNG seed retained for any future stochastic step; spatial matching is deterministic.
RNG_SEED = 0

## Helper functions

All helpers operate on 3D arrays `(T, Y, X)`. `to_3d` reduces the 5D `(T, C, Z, Y, X)` array returned by bioio. Each metric function returns a per-frame `(T,)` trajectory of mean actin intensity over the population it defines.

In [ ]:
def to_3d(arr: np.ndarray) -> np.ndarray:
    """
    Squeeze a (T, Z, Y, X) or (T, 1, Z, Y, X) array down to (T, Y, X).
    Errors if the result is not 3D.
    """
    a = np.asarray(arr)
    a = a.squeeze()
    if a.ndim != 3:
        raise ValueError(f'Expected (T, Y, X) after squeeze; got shape {a.shape}')
    return a


def _frame_mean_with_mask(values: np.ndarray, mask: np.ndarray) -> np.ndarray:
    """
    Per-frame mean of `values` over pixels where `mask` is True.
    Returns NaN for frames with no contributing pixels.
    Both inputs are (T, Y, X). `mask` must be boolean.
    """
    if mask.dtype != bool:
        mask = mask.astype(bool)
    T = values.shape[0]
    flat_mask = mask.reshape(T, -1)
    counts = flat_mask.sum(axis=1)
    out = np.full(T, np.nan, dtype=float)
    has = counts > 0
    if has.any():
        sums = (values.astype(float) * mask).reshape(T, -1).sum(axis=1)
        out[has] = sums[has] / counts[has]
    return out

In [ ]:
def actin_compact_nextframe(actin: np.ndarray, seg_cmp: np.ndarray) -> np.ndarray:
    """
    Mean actin at frame t over pixels noncompact at t and compact at t+1
    (0 -> 1 transition). Last frame is NaN. Shape: (T,).
    """
    seg_cmp = seg_cmp.astype(bool)
    mask = np.zeros_like(seg_cmp)
    mask[:-1] = ~seg_cmp[:-1] & seg_cmp[1:]
    return _frame_mean_with_mask(actin, mask)


def actin_in_noncompact_nextframe(actin: np.ndarray, seg_caax: np.ndarray) -> np.ndarray:
    """
    Mean actin at frame t over pixels noncompact (CAAX+) at both t and t+1
    (0 -> 0 persistence). Last frame is NaN. Shape: (T,).
    """
    seg_caax = seg_caax.astype(bool)
    mask = np.zeros_like(seg_caax)
    mask[:-1] = seg_caax[:-1] & seg_caax[1:]
    return _frame_mean_with_mask(actin, mask)

In [ ]:
def leading_up_to_cmp(
    actin: np.ndarray,
    seg_cmp: np.ndarray,
    seg_caax: np.ndarray,
    lookback: int = 5,
    neighbor_radius_px: float = 50.0,
    rng_seed: int | None = 0,  # kept for API stability; spatial matching is deterministic
):
    """
    Per-image actin trajectory aligned to first-compaction events.

    Compactor cohort: pixels that ever compact and have at least `lookback`
    frames of history before their first compaction. Restricting to this cohort
    keeps the contributing pixel population identical across all relative-time
    slots, so trajectory shape reflects within-pixel temporal change rather than
    cohort-composition drift.

    Never compact control (spatially matched): always-CAAX+ pixels that never
    compact AND lie within `neighbor_radius_px` of some compactor pixel that
    passes the lookback filter (Euclidean distance, pixel units). Each such
    control inherits the `first_cmp` time of its single nearest valid compactor
    as its pseudo-event time. This gives a *local* reference that controls for
    illumination/focal-plane gradients and position-dependent biology, while
    still being aligned to the same actual frame numbers as a real compaction
    event (controls for photobleaching, drift, culture-age effects).

    Restricting the matching target to lookback-passing compactors guarantees
    every control pixel has a pseudo-event time >= lookback, so the control
    cohort is also constant across all relative-time slots.

    Returns
    -------
    compactor : ndarray, shape (T,)
        Trajectory aligned by row index to the dataframe rows of this image.
        Row t holds the value at relative time r = t - (T - 1). Values are
        NaN outside r in [-lookback, 0].
    control : ndarray, shape (T,)
        Same alignment, for the spatially matched never compact control cohort.
    n_compactor : int
        Number of qualifying compactor pixels (constant across relative time).
    n_control : int
        Number of qualifying control pixels (constant across relative time).
    """
    T = actin.shape[0]
    seg_cmp = seg_cmp.astype(bool)
    seg_caax = seg_caax.astype(bool)

    ever_compacts = seg_cmp.any(axis=0)                                 # (Y, X)
    first_cmp = np.where(ever_compacts, seg_cmp.argmax(axis=0), -1)     # (Y, X)

    # Compactor cohort: lookback-passing first-compaction pixels.
    valid = ever_compacts & (first_cmp >= lookback)
    py, px = np.where(valid)

    # Control candidates: always-CAAX pixels that never compact.
    candidate = seg_caax.all(axis=0) & ~ever_compacts                   # (Y, X)

    # Spatial matching: each candidate -> nearest *valid* compactor pixel.
    # distance_transform_edt computes, for each TRUE pixel in its input, the
    # Euclidean distance (and indices) to the nearest FALSE pixel. Passing
    # ~valid makes "FALSE" = compactor, so distance/idx encode "distance to
    # nearest valid compactor" and "(yy, xx) of that compactor".
    if valid.any():
        distance, idx = distance_transform_edt(
            ~valid, return_distances=True, return_indices=True,
        )
        in_radius = candidate & (distance <= neighbor_radius_px)
        cy, cx = np.where(in_radius)
        if len(cy):
            ny = idx[0][cy, cx]
            nx = idx[1][cy, cx]
            pseudo_t = first_cmp[ny, nx]
        else:
            pseudo_t = np.array([], dtype=int)
    else:
        cy = np.array([], dtype=int)
        cx = np.array([], dtype=int)
        pseudo_t = np.array([], dtype=int)

    compactor = np.full(T, np.nan, dtype=float)
    control = np.full(T, np.nan, dtype=float)

    last_t = T - 1
    for r in range(-lookback, 1):
        row = last_t + r                       # row whose relative-time label corresponds to r
        if len(py):
            t_c = first_cmp[py, px] + r
            compactor[row] = float(actin[t_c, py, px].mean())
        if len(cy):
            t_nc = pseudo_t + r
            ok = (t_nc >= 0) & (t_nc < T)
            if ok.any():
                control[row] = float(actin[t_nc[ok], cy[ok], cx[ok]].mean())

    return compactor, control, int(len(py)), int(len(cy))


def leading_up_to_uncmp(actin, seg_cmp, lookback=1):
    """
    Per-image actin trajectory aligned to first-uncompaction events (1 -> 0).

    For each pixel that ever uncompacts, find the frame of its first 1 -> 0
    transition. Read actin backwards over `lookback + 1` slots (relative
    times -lookback ... 0). Default lookback = 1 frame, so the trajectory
    holds two values per pixel: actin one frame before uncompaction (still
    compact) and actin at the uncompaction frame (now noncompact).

    Pixels whose first uncompaction is earlier than `lookback` frames in are
    excluded so the contributing cohort is identical at every relative-time
    slot.

    Returns
    -------
    trajectory : ndarray (T,)
        Per-frame mean actin, aligned by row index. Row T-1+r holds the
        value at relative time r for r in [-lookback, 0]. NaN elsewhere.
    n_pixels : int
        Number of qualifying pixels (constant across relative time).
    """
    T = actin.shape[0]
    seg_cmp = seg_cmp.astype(bool)

    # 1 -> 0 transitions: at frame t, pixel was compact at t-1 and noncompact at t.
    uncmp_events = np.zeros_like(seg_cmp)
    uncmp_events[1:] = seg_cmp[:-1] & ~seg_cmp[1:]

    ever_uncmp = uncmp_events.any(axis=0)
    first_uncmp = np.where(ever_uncmp, uncmp_events.argmax(axis=0), -1)  # (Y, X)

    valid = ever_uncmp & (first_uncmp >= lookback)
    py, px = np.where(valid)

    trajectory = np.full(T, np.nan, dtype=float)
    last_t = T - 1
    for r in range(-lookback, 1):
        row = last_t + r
        if len(py):
            t_u = first_uncmp[py, px] + r
            trajectory[row] = float(actin[t_u, py, px].mean())

    return trajectory, int(len(py))

## Main loop

Iterate over unique ROI images. For each image: load channels, compute per-frame and event-aligned metrics, write them to the corresponding `df` rows, and save a derived-mask OME-TIFF for visual QC. The dataframe is checkpointed to disk after each image so partial progress survives interruption.

In [ ]:
df = pd.read_csv(df_path)
df.head()

In [ ]:
imgpaths = (input_dirpath / df['ROI imgname']).unique()

for imgpath in tqdm(imgpaths):
    imgpath = Path(imgpath)
    if not imgpath.is_file():
        print(f'SKIP (not found): {imgpath.name}')
        continue

    # ---- Load image and extract channels ----
    img_file = BioImage(imgpath, reader=bioio_ome_tiff.Reader)
    img = img_file.data                                         # (T, C, Z, Y, X)

    actin = to_3d(img[:, ACTIN_CH]).astype(float)               # (T, Y, X)
    seg_cmp = to_3d(img[:, SEG_CMP_CH]) > 0                     # (T, Y, X) bool, compact pixels
    seg_caax = to_3d(img[:, SEG_CAAX_CH]) > 0                   # (T, Y, X) bool, noncompact pixels
    T = actin.shape[0]

    rows = df.index[df['ROI imgname'] == imgpath.name].tolist()
    if len(rows) != T:
        print(f'WARNING: {imgpath.name} has {T} frames but {len(rows)} dataframe rows')

    # ---- Per-image time interval -> lookback in frames ----
    dt = float(df.loc[rows[0], 'time interval'])               # hours per frame
    lookback_frames = int(round(LOOKBACK_TIME / dt))
    if lookback_frames < 1:
        print(f'WARNING: {imgpath.name}: LOOKBACK_TIME={LOOKBACK_TIME}hr < dt={dt}hr; clamping to 1 frame')
        lookback_frames = 1

    # ---- Per-frame metrics on noncompact (CAAX+) pixels ----
    df.loc[rows, 'mean actin int, compact next frame'] = (
        actin_compact_nextframe(actin, seg_cmp)
    )
    df.loc[rows, 'mean actin int, noncompact next frame'] = (
        actin_in_noncompact_nextframe(actin, seg_caax)
    )

    # ---- Leading-up-to-compaction trajectory ----
    df.loc[rows, 'time relative to cmp (hr)'] = np.arange(-T + 1, 1) * dt
    cmp_traj, ctrl_traj, n_cmp, n_ctrl = leading_up_to_cmp(
        actin, seg_cmp, seg_caax,
        lookback=lookback_frames,
        neighbor_radius_px=NEIGHBOR_RADIUS_PX,
        rng_seed=RNG_SEED,
    )
    df.loc[rows, 'mean actin int leading up to cmp'] = cmp_traj
    df.loc[rows, 'mean actin int leading up to cmp (never compact control)'] = ctrl_traj
    df.loc[rows, 'n pixels (compactor)'] = n_cmp
    df.loc[rows, 'n pixels (never compact)'] = n_ctrl

    # ---- Leading-up-to-uncompaction trajectory (1 -> 0 events) ----
    df.loc[rows, 'time relative to uncmp (hr)'] = np.arange(-T + 1, 1) * dt
    uncmp_traj, n_uncmp = leading_up_to_uncmp(actin, seg_cmp, lookback=1)
    df.loc[rows, 'mean actin int leading up to uncmp'] = uncmp_traj
    df.loc[rows, 'n pixels (uncompactor)'] = n_uncmp

    # ---- Save derived-mask OME-TIFF for visual QC ----
    mask_compact_nextframe = np.zeros_like(seg_cmp)
    mask_compact_nextframe[:-1] = ~seg_cmp[:-1] & seg_cmp[1:]                 # 0 -> 1
    mask_noncompact_nextframe = np.zeros_like(seg_caax)
    mask_noncompact_nextframe[:-1] = seg_caax[:-1] & seg_caax[1:]             # 0 -> 0
    cumul_compact = np.cumsum(seg_cmp.astype(int), axis=0) > 0
    mask_first_compact = np.zeros_like(seg_cmp)
    mask_first_compact[1:] = cumul_compact[1:] & ~cumul_compact[:-1]          # first compact frame per pixel
    mask_always_noncompact = np.broadcast_to(
        seg_caax.all(axis=0)[None], seg_caax.shape
    )
    mask_uncompact_nextframe = np.zeros_like(seg_cmp)
    mask_uncompact_nextframe[:-1] = seg_cmp[:-1] & ~seg_cmp[1:]               # 1 -> 0

    # Re-add C and Z singleton dims for OME-TIFF (T, C, Z, Y, X)
    def _expand(a):
        return a.astype('uint16')[:, None, None, :, :]
    stack = np.concatenate([
        _expand(actin), _expand(seg_cmp), _expand(seg_caax),
        _expand(mask_compact_nextframe), _expand(mask_noncompact_nextframe),
        _expand(mask_first_compact), _expand(mask_always_noncompact),
        _expand(mask_uncompact_nextframe),
    ], axis=1)
    ome_metadata = utils.construct_ome_metadata(stack, img_file)
    OmeTiffWriter.save(stack, output_dirpath / imgpath.name, ome_xml=ome_metadata)

    # ---- Checkpoint ----
    df.to_csv(df_path, index=False)

print('Done!')

## Per-cell normalization


In [ ]:
# Per-cell normalization. Each actin column is divided by its cell's mean LifeAct
# intensity (averaged across all frames of that cell). Normalized columns are
# written with a `norm ` prefix; original columns are preserved.

norm_ref = df.groupby('UID')['mean actin int (cell)'].transform('mean')

actin_cols = [
    c for c in df.columns
    if 'actin' in c.lower()
    and not c.startswith('norm ')
    and c != 'mean actin int (cell)'   # skip the reference itself
]

df[[f'norm {c}' for c in actin_cols]] = df[actin_cols].div(norm_ref, axis=0)

df.to_csv(df_path, index=False)
df.head()

## Event-aligned trajectories

Mean actin intensity (per-cell-normalized) read backwards from each pixel’s first-compaction event for compactor pixels and from the matched pseudo-event time for the spatially matched never-compact control. The second plot shows the paired actin values at the frame before and the frame of first uncompaction.

In [ ]:
# Leading-up-to-cmp: trajectory aligned to first-compaction events (per-cell-normalized).
# Compactor: pixels that compact at relative time 0.
# Control:   spatially matched always-noncompact pixels paired in time with
#            their nearest valid compactor.
xcol = 'time relative to cmp (hr)'
fig, ax = plt.subplots(figsize=(4, 3))
sns.lineplot(df, x=xcol, y='norm mean actin int leading up to cmp',
             label='compactor', errorbar='se', ax=ax)
sns.lineplot(df, x=xcol, y='norm mean actin int leading up to cmp (never compact control)',
             label='never compact (matched)', errorbar='se', ax=ax)
ax.set_ylabel('normalized actin intensity')
ax.legend(frameon=False)
plt.tight_layout()
plt.show()

# Leading-up-to-uncmp: actin one frame before uncompaction vs at uncompaction.
fig, ax = plt.subplots(figsize=(4, 3))
sns.lineplot(df, x='time relative to uncmp (hr)',
             y='norm mean actin int leading up to uncmp',
             marker='o', errorbar='se', ax=ax)
ax.set_ylabel('normalized actin intensity')
plt.tight_layout()
plt.show()